# ****BUSINESS UNDERSTANDING****

<p align="center">
    <img src="../xLain-Lain/img/bu_2.png" alt="Pendahuluan" witdh="100" height="500">
</p>

### ****DAFTAR ISI****

1. Tujuan dan Posisi Notebook dalam Alur CRISP-DM
2. Latar Belakang dan Konteks Bisnis
3. Perumusan Masalah
4. Tujuan Bisnis -> Tujuan Data Mining
5. Kriteria Keberhasilan
6. Landasan Teori dan Hipotesis Ekonomi
7. Analisis Situasi: Sumber Daya, Asumsi, Batasan, dan Risiko
8. Pemetaan Rublik Penilaian
9. Skema Prediksi dan Prinsip Anti-Leakage
10. Rencana Proyek dan Jadwal
11. Konfigurasi Proyek dan Uji Kesiapan
12. Kesimpulan dan Langkah Berikutnya
13. Referensi

---

### ****1. Tujuan dan Posisi Notebook dalam Alur CRISP-DM****

CRISP-DM (Chapman et al., 2000) adalah kerangka proses data mining yang bersifat iteratif: hasil di satu fase dapat mengembalikan pekerjaan ke fase sebelumnya. Kerangka ini dipilih karena: menjadi standar de facto lintas industri; memaksa pemisahan yang tegas antara memahami masalah, memahami data, dan membangun model; selaras dengan struktur folder proyek tim.

Tujuan notebook ini:

1. Menerjemahkan narasi studi kasus menjadi **masalah prediksi yang terdefinisi secara formal**.
2. Menetapkan **kriteria keberhasilan yang terukur** sebelum melihat hasil model.
3. Merumuskan **hipotesis ekonomi yang dapat diuji** sebagai dasar rekayasa fitur (Task 1).
4. Mengidentifikasi **risiko** utama beserta mitigasinya, terutama risiko kebocoran data (Task 2) dan pergeseran rezim pasar (Task 3).
5. Menyusun rencana kerja yang realistis terhadap tenggat.

Gambar di bawah ini menggambarkan peta jalan proyek.

<p align="center">
    <img src="../xLain-Lain/img/bu_4.png" alt="Pendahuluan" witdh="100" height="400">
</p>

*Fase *Deployment* pada CRISP-DM standar tidak dilakukan secara nyata (di luar lingkup lomba), tetapi kami mengisinya dengan cetak biru Sistem Peringatan Dini dan rekomendasi bagi manajemen risiko.*

### ****2. Latar Belakang dan Konteks Bisnis****

**PT Investion** adalah pengelola aset dengan dana kelolaan bernilai triliunan rupiah yang sangat terpapar pada pergerakan Indeks Harga Saham Gabungan (IHSG). Studi kasus menyebut empat saluran transmisi guncangan eksternal ke pasar saham domestik:

| Saluran | Variabel pada Dataset | Mekanisme Ekonomi |
|---|---|---|
| Sentimen risiko global | `VIX`, `SP500` | Lonjakan ketakutan investor global memicu penarikan dana dari pasar negara berkembang (*risk-off*) |
| Kurs | `USDIDR` | Depresiasi rupiah menggerus imbal hasil investor asing dalam mata uang dasar dan memicu *capital outflow* |
| Komoditas | `OIL`, `GOLD` | Minyak mencerminkan siklus ekonomi dan biaya input; emas berfungsi sebagai aset *safe haven* |
| Kebijakan moneter | `BI_rate`, `US_rate`, `CPI` | Selisih suku bunga menentukan daya tarik aset rupiah; inflasi memengaruhi ekspektasi kebijakan |

Guncangan tersebut tidak berdampak sesaat. Pasar keuangan menunjukkan *Volatility Clustering*, yakni periode bergejolak cenderung diikuti periode bergejolak lain (Mandelbrot, 1963; Engle, 1982). Akibatnya, pengelolaan risiko yang hanya mengandalkan tren harga historis kurang memadai: arah harga sulit diprediksi, tetapi besarnya gejolak jauh lebih dapat diprediksi karena persistensinya (Bollerslev, 1986; Corsi, 2009).

<table>
  <tr>
    <td><img src="../xLain-Lain/img/bu_5.png" width="400" alt="Mandelbrot"></td>
  </tr>
</table>

Dua tuntutan yang membedakan proyek ini dari proyek prediksi biasa:

1. Peringatan dini, bukan sekedar prakiraan. Nilai bisnisnya ada pada kemampuan sinyal sebelum gejolak terjadi.
2. Tanpa *black box*. Komite risiko menuntut pertanggungjawaban logis atas setiap prediksi, sehingga model harus dapat dibongkar dengan XAI.

### ****3. Perumusan Masalah (Formal)****

**Pertanyaan Bisnis**. *Seberapa bergejolak IHSG dalam 5 hari perdagangan ke depan dan faktor ekonomi apa yang menjelaskan prediksi tersebut?*

**Formalisasi**. Misalkan $P_t$ adalah harga penutupan IHSG pada hari perdagangan $t$ dan $r_t = P_t/P_{t-1} - 1$ adalah *return* hariannya. Misalkan $\mathcal{F}_t$ adalah seluruh informasi yang tersedia setelah penutupan hari $t$. Hipotesis kerja untuk target adalah

$$
Y_t \;=\; \sigma_{t,\,t+5} \;=\; \sqrt{\frac{1}{4}\sum_{i=1}^{5}\left(r_{t+i}-\bar r_{t,5}\right)^2},\qquad \bar r_{t,5}=\frac{1}{5}\sum_{i=1}^{5} r_{t+i}.
$$

Tugas pemodelan adalah mempelajari fungsi

$$
\hat Y_t = f\big(\mathbf{x}_t\big),\qquad \mathbf{x}_t \in \mathcal{F}_t,
$$

yang meminimalkan galat prediksi (RMSE/MAE, serta QLIKE sebagai pelengkap) di bawah kendala kausalitas: $\mathbf{x}_t$ tidak boleh memuat informasi apa pun yang baru diketahui pada $t+1$ atau sesudahnya.

**Catatan penting.** Karena $Y_t$ sendiri dibangun dari *return* masa depan ($t+1,\dots,t+5$), kebocoran data dapat terjadi secara halus, misalnya lewat *lag* target yang masih tumpang tindih dengan jendela masa depan. Hal ini dianalisis tuntas di Notebook 2 dan diatasi di Notebook 3.

**Jenis masalah:** regresi deret waktu univariat-target dengan prediktor multivariat eksogen dan horizon tetap $h=5$ hari. **Unit observasi:** satu hari perdagangan. **Data uji:** seluruh baris sejak 2025 (nilai $Y$ tidak tersedia).

### ****4. Tujuan Bisnis -> Tujuan Data Mining****

| # | Tujuan Bisnis (PT Investion) | Tujuan Data Mining yang Menerjemahkannya | Bukti yang akan Ditampilkan |
|---|---|---|---|
| BO1 | Mendapat indikasi dini kondisi pasar berisiko | *DM1*: model regresi $\hat Y_t$ untuk volatilitas 5 hari ke depan<br>*DM2*: konversi $\hat Y_t$ ke level peringatan Hijau/Kuning/Merah berbasis kuantil data latih | RMSE/MAE/QLIKE vs. baseline; presisi-*recall* kejadian volatilitas tinggi |
| BO2 | Setiap prediksi dapat dipertanggungjawabkan | *DM3:* XAI global (SHAP summary), lokal (*waterfall*), dan analisis kesalahan berbasis XAI | Grafik SHAP; narasi bisnis tiap fitur |
| BO3 | Sistem dapat dipercaya (tanpa informasi "dari masa depan") | *DM4:* pemisahan *walk-forward* dengan *purge gap*, *pipeline* yang di-*fit* hanya pada data latih, dan uji kausalitas fitur | Uji pemotongan data (*truncation test*) lulus 100% |
| BO4 | Sistem tetap andal saat rezim pasar berubah (2025–2026) | *DM5:* fitur stasioner, analisis *drift* latih vs uji, evaluasi pada fold bertekanan | Uji ADF/KPSS, *adversarial validation*, kinerja per rezim |

### ****5. Kriteria Keberhasilan****

Kenapa kami tetapkan kriteria sebelum pemodelan? Agar penilaian tidak dibelokkan oleh hasil yang kebetulan baik.

**A. Kriteria Teknis (kuantitatif)**

| # | Kriteria | Ukuran |
|---|---|---|
| T1 | Model ML lebih akurat daripada baseline naif (rata-rata volatilitas 5 hari ke belakang) | RMSE dan MAE lebih kecil pada validasi *walk-forward*, secara konsisten di mayoritas fold |
| T2 | Keunggulan bersifat signifikan secara statistik | Uji Diebold-Mariano (1995) dengan $p<0{,}05$ pada galat kuadrat/absolut |
| T3 | Model bersaing dengan *benchmark* ekonometrik standar | Dibandingkan dengan HAR-OLS (Corsi, 2009) dan GARCH(1,1) (Bollerslev, 1986); aspirasi: setara atau lebih baik |
| T4 | Tidak ada kebocoran data | Uji kausalitas: fitur pada hari $t$ identik bila data setelah $t$ dihapus; fit *scaler/imputer* hanya pada data latih |
| T5 | Tuning bersih | Seluruh pencarian hyperparameter hanya memakai data latih dan validasi |
| T6 | Penjelasan stabil | Peringkat fitur SHAP konsisten antar fold dan sejalan dengan *permutation importance* |

**B. Kriteria Bisnis (kualitatif)**

| # | Kriteria |
|---|---|
| B1 | Setiap fitur utama memiliki logika ekonomi eksplisit yang dapat dijelaskan kepada komite risiko |
| B2 | Penjelasan lokal dapat diterjemahkan ke bahasa bisnis untuk tanggal berisiko tinggi |
| B3 | Kelemahan model (kapan ia meleset dan mengapa) diungkapkan secara jujur |

**C. Kriteria Kompetisi**

| # | Kriteria |
|---|---|
| K1 | Notebook `DSC_apa ya namanya, bingung juga_Notebook.ipynb` menjawab seluruh Task 1–3 |
| K2 | Slide `DSC_apa ya namanya, bingung juga_PPT.pptx` berbahasa Indonesia, tanpa unsur terlarang |
| K3 | Kode orisinal; tautan Drive diunggah sebelum 29 Oktober 2026 pukul 23.59 WIB |

*Kami sengaja tidak menetapkan target angka RMSE absolut. Tingkat kesulitan prediksi volatilitas bergantung pada rezim pasar, sehingga angka absolut tanpa konteks mudah menyesatkan. Ukuran yang bermakna adalah perbaikan relatif terhadap baseline pada protokol validasi yang sama*


### ****6. Landasan Teori dan Hipotesis Ekonomi****

**6.1 Volatilitas Sebagai Besaran yang Dapat Diprediksi**

Harga aset hampir tidak dapat diprediksi arahnya, tetapi kuadrat atau nilai mutlak return menunjukkan autokorelasi positif yang kuat dan bertahan lama. Inilah dasar keluarga model ARCH/GARCH (Engle, 1982; Bollerslev, 1986). Hansen & Lunde (2005) menunjukkan bahwa GARCH(1,1) sangat sulit dikalahkan secara konsisten, sehingga ia menjadi benchmark yang adil.

**6.2 Struktur Multi-Horizon (HAR)**

Model HAR-RV (Corsi, 2009) menjelaskan volatilitas ke depan sebagai kombinasi linear volatilitas realisasi harian, mingguan, dan bulanan, yang mencerminkan perilaku pelaku pasar dengan horizon investasi berbeda. Komponen volatilitas jangka pendek dan menengah dari IHSG hampir pasti menjadi prediktor penting, sehingga fitur kami harus memberi informasi di luar apa yang sudah ditangkap komponen tersebut.

**6.3 Leverage Effect**

Guncangan negatif cenderung menaikkan volatilitas lebih besar daripada guncangan positif berukuran sama (Black, 1976; Engle & Ng, 1993). Implikasinya, ukuran volatilitas turun (*downside semivariance*) layak diuji sebagai fitur.

**6.4 Transmisi Lintas Pasar**

- Ketakutan Global. VIX adalah ukuran volatilitas implisit S&P 500 yang dikenal sebagai *investor fear gauge* (Whaley, 2000). Selisih antara ekspektasi volatilitas (implisit) dan volatilitas yang terealisasi dikenal sebagai *variance risk premium* dan memiliki kekuatan prediktif atas pergerakan pasar (Bollerslev, Tauchen & Zhou, 2009).
- Siklus Keuangan Global. Rey (2013) menunjukkan bahwa siklus keuangan global (tercermin pada VIX dan kebijakan moneter AS) memengaruhi kondisi keuangan negara berkembang terlepas dari rezim kurs mereka. Forbes & Warnock (2012) mendokumentasikan gelombang arus modal ke negara berkembang yang dipicu faktor global.
- Emas berfungsi sebagai *safe haven* saat pasar tertekan (Baur & Lucey, 2010); Minyak adalah penanda siklus ekonomi dan guncangan permintaan/penawaran (Hamilton, 2009).

**6.5 Daftar Hipotesis yang Akan Diuji**

Hipotesis berikut menjadi dasar kandidat fitur pada Task 1. Tiga fitur terbaik dipilih berdasarkan bukti EDA dan validasi.


| ID | Kandidat Fitur | Mekanisme Ekonomi | Tanda Diharapkan | Bukti yang Diminta |
| :--- | :--- | :--- | :---: | :--- |
| H1 | Variance Risk Premium proxy<br>*(VIX terskala - RV IHSG)* | Ketakutan global yang belum terserap pasar domestik akan menyusul sebagai lonjakan volatilitas IHSG | + | Korelasi Spearman, SHAP dependence |
| H2 | Rupiah Stress Index<br>*(z-score return USDIDR x spread BI-US)* | Depresiasi rupiah + selisih bunga menyempit memicu arus modal keluar, menaikkan volatilitas ekuitas | + | Korelasi, importance, SHAP lokal |
| H3 | Rasio volatilitas jangka pendek/menengah<br>*(RV5 / RV22)* | Volatilitas melonjak jauh di atas rata-rata bulanan cenderung kembali ke rata-rata (*mean reversion*) | +/- | Korelasi, SHAP dependence non-linear |
| H4 | Downside semivariance IHSG | Guncangan negatif menaikkan volatilitas lebih besar daripada guncangan positif (*leverage effect*) | + | Perbandingan dengan RV simetris |
| H5 | Guncangan emas dan minyak<br>*( \|return\|, z-score)* | Gejolak komoditas menandakan perubahan sentimen/permintaan dan perpindahan ke *safe haven* | + | Korelasi, importance |
| H6 | Perubahan VIX 1 dan 5 hari | Percepatan ketakutan global lebih informatif daripada levelnya | + | Korelasi, SHAP | NB3, NB5 |
| H7 | Hari sejak perubahan suku bunga BI / The Fed | Guncangan kebijakan menimbulkan ketidakpastian sementara yang meluruh seiring waktu | - | Dependence plot |